In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from scripts.helpers import save_classification_report

RANDOM_STATE = 42

In [4]:
df = pd.read_csv(PROJECT_ROOT / "working_data" / "nhamcs_data_2018_22.csv")


In [5]:
import numpy as np

print("Applying Cyclical Encoding to Time Features...")

# 1. Clean and convert arrival_time (HHMM integer) to a continuous hour scale (0 to 23.99)
# Example: 1430 -> 14 hours + (30/60) minutes = 14.5
if 'arrival_time' in df.columns:
    # Set missing or invalid times (often negative in NHAMCS) to NaN    
    df['arrival_time'] = np.where(df['arrival_time'] < 0, np.nan, df['arrival_time'])
    
    # Extract hours and minutes
    hours = np.floor_divide(df['arrival_time'], 100)
    minutes = np.mod(df['arrival_time'], 100)
    
    # Create the continuous hour feature
    df['arrival_hour_float'] = hours + (minutes / 60.0)
    df['arrival_hour'] = hours
    
    # Shift overlap flag (06:00-08:00, 18:00-20:00)
    df['is_shift_change'] = (
        ((df['arrival_hour_float'] >= 6) & (df['arrival_hour_float'] < 8))
        | ((df['arrival_hour_float'] >= 18) & (df['arrival_hour_float'] < 20))
    ).astype(int)

# 2. Define the exact maximum values for a full cycle
cycle_maxes = {
    'visit_month': 12.0,
    'day_of_week': 7.0,
    'arrival_hour_float': 24.0
}

# 3. Apply the Sine and Cosine Transformations
for col, max_val in cycle_maxes.items():
    if col in df.columns:
        # Calculate the angle on the circle
        angle = 2 * np.pi * df[col] / max_val
        
        # Create the Sin and Cos features
        df[f'{col}_sin'] = np.sin(angle)
        df[f'{col}_cos'] = np.cos(angle)

# Weekend/weeknight interaction
if 'day_of_week' in df.columns:
    max_dow = df['day_of_week'].max()
    weekend_days = [5, 6] if max_dow <= 6 else [6, 7]
    is_weekend = df['day_of_week'].isin(weekend_days)
    if 'arrival_hour_float' in df.columns:
        df['is_weekend_night'] = (is_weekend & (df['arrival_hour_float'] >= 18)).astype(int)
    else:
        df['is_weekend'] = is_weekend.astype(int)

# 4. Drop the original linear time columns to prevent multicollinearity
cols_to_drop = ['visit_month', 'day_of_week', 'arrival_time', 'arrival_hour_float']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Dataframe shape after cyclical encoding: {df.shape}")
print("\nSample of the new Time Features:")
display(df[[c for c in df.columns if '_sin' in c or '_cos' in c]].head())

Applying Cyclical Encoding to Time Features...
Dataframe shape after cyclical encoding: (58124, 55)

Sample of the new Time Features:


,visit_month_sin,visit_month_cos,day_of_week_sin,day_of_week_cos,arrival_hour_float_sin,arrival_hour_float_cos
0,-2.449294e-16,1.000000,0.781831,0.623490,-0.748956,0.662620
1,-2.449294e-16,1.000000,0.781831,0.623490,-0.957571,0.288196
2,-2.449294e-16,1.000000,-0.781831,0.623490,-0.625923,-0.779884
3,-2.449294e-16,1.000000,-0.433884,-0.900969,-0.994522,-0.104528
4,-5.000000e-01,0.866025,0.974928,-0.222521,-0.983255,-0.182236


In [6]:
print("Adding clinical ratios and vital missingness...")

df["shock_index"] = df["heart_rate"] / df["sys_bp"].replace(0, np.nan)
df["shock_index_age_adj"] = df["shock_index"] * np.where(df["age"] >= 65, 1.2, 1.0)

df["map"] = (df["sys_bp"] + 2 * df["dias_bp"]) / 3
df["pulse_pressure"] = df["sys_bp"] - df["dias_bp"]

df["age_hr_interaction"] = df["age"] * df["heart_rate"]

df["resp_spo2_ratio"] = df["resp_rate"] / df["spo2"].replace(0, np.nan)

df["elderly_tachy"] = ((df["age"] >= 65) & (df["heart_rate"] > 100)).astype(int)

history_cols = [c for c in df.columns if any(k in c for k in "hist_")]
if history_cols:
    hist_numeric = df[history_cols].apply(pd.to_numeric, errors="coerce")
    df["history_count"] = hist_numeric.fillna(0).sum(axis=1)

# NEWS2-style score (approximate)
news2 = pd.Series(0, index=df.index, dtype=float)
if "resp_rate" in df.columns:
    rr = df["resp_rate"]
    news2 += np.select([
        rr <= 8,
        (rr >= 9) & (rr <= 11),
        (rr >= 12) & (rr <= 20),
        (rr >= 21) & (rr <= 24),
        rr >= 25,
    ], [3, 1, 0, 2, 3], default=0)
if "spo2" in df.columns:
    sp = df["spo2"]
    news2 += np.select([
        sp <= 91,
        (sp >= 92) & (sp <= 93),
        (sp >= 94) & (sp <= 95),
        sp >= 96,
    ], [3, 2, 1, 0], default=0)
if "temp" in df.columns:
    t = df["temp"]
    news2 += np.select([
        t <= 35.0,
        (t > 35.0) & (t <= 36.0),
        (t > 36.0) & (t <= 38.0),
        (t > 38.0) & (t <= 39.0),
        t > 39.0,
    ], [3, 1, 0, 1, 2], default=0)
if "sys_bp" in df.columns:
    sbp = df["sys_bp"]
    news2 += np.select([
        sbp <= 90,
        (sbp >= 91) & (sbp <= 100),
        (sbp >= 101) & (sbp <= 110),
        (sbp >= 111) & (sbp <= 219),
        sbp >= 220,
    ], [3, 2, 1, 0, 3], default=0)
if "heart_rate" in df.columns:
    hr = df["heart_rate"]
    news2 += np.select([
        hr <= 40,
        (hr >= 41) & (hr <= 50),
        (hr >= 51) & (hr <= 90),
        (hr >= 91) & (hr <= 110),
        (hr >= 111) & (hr <= 130),
        hr >= 131,
    ], [3, 1, 0, 1, 2, 3], default=0)

mental_col = next((c for c in ["mental_status", "mental_status_history", "avpu", "arrem", "arrem_score"] if c in df.columns), None)
if mental_col is not None:
    ms = df[mental_col]
    if pd.api.types.is_numeric_dtype(ms):
        is_alert = ms.fillna(0) == 0
    else:
        ms_str = ms.astype(str).str.lower()
        is_alert = ms_str.str.contains("alert") | (ms_str == "a")
    news2 += np.where(is_alert, 0, 3)
df["news2_score"] = news2

pain_col = next((c for c in ["pain_score", "pain_scale", "pain"] if c in df.columns), None)
if pain_col is not None and mental_col is not None:
    pain_high = df[pain_col] >= 7
    df["pain_ams_flag"] = (pain_high & (~is_alert)).astype(int)

vital_cols = ["heart_rate", "sys_bp", "dias_bp", "resp_rate", "spo2", "temp"]
vital_cols = [c for c in vital_cols if c in df.columns]
if vital_cols:
    df["vital_missing"] = df[vital_cols].isna().any(axis=1).astype(int)
    for col in vital_cols:
        df[f"{col}_missing"] = df[col].isna().astype(int)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

Adding clinical ratios and vital missingness...


In [7]:
# Prepare features and target for ordinal triage
cols_to_drop = ["intervention_iv_fluids", "vitals_during_visit", "wait_time_minutes", "year", "target_triage_acuity", "chief_complaint_text", "injury_cause_text"]
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns]).copy()
y = df["target_triage_acuity"] - 1

cat_cols = X.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    X[col] = X[col].astype("category")

In [8]:
from sklearn.metrics import cohen_kappa_score

def qwk_score(y_true, y_pred):
    """Quadratic Weighted Kappa for ordinal triage evaluation."""
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


In [12]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.ensemble import BalancedRandomForestClassifier
from lightgbm import LGBMClassifier

xgb_params = {
    "n_estimators": 2000,
    "max_depth": 6,
    "learning_rate": 0.01,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "eval_metric": "mlogloss",
    "enable_categorical": True,
    "tree_method": "hist",
}
rf_params = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "class_weight": "balanced",
}
lgbm_params = {
    "n_estimators": 2000,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multiclass",
    "random_state": RANDOM_STATE,
    "class_weight": "balanced",
    "n_jobs": -1,
    "verbose": -1,
}

In [10]:
# Holdout evaluation (train/val split)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = XGBClassifier(**xgb_params)
sample_weights = compute_sample_weight("balanced", y_train)
model.fit(X_train, y_train, sample_weight=sample_weights)

holdout_pred = model.predict(X_val)
holdout_qwk = qwk_score(y_val, holdout_pred)
print(f"Holdout QWK: {holdout_qwk:.4f}")

save_classification_report(
    y_val,
    holdout_pred,
    model_name="xgb_weighted_tabular_holdout",
    seed=RANDOM_STATE,
    config="xgb_weighted",
    columns=X_train.columns,
    notes="weighted XGB tabular holdout",
    extra_metrics={"qwk": holdout_qwk},
)

Holdout QWK: 0.3659


{'0': {'precision': 0.18303571428571427,
  'recall': 0.24260355029585798,
  'f1-score': 0.20865139949109415,
  'support': 169.0},
 '1': {'precision': 0.33423493044822256,
  'recall': 0.5031995346131471,
  'f1-score': 0.401671697237056,
  'support': 1719.0},
 '2': {'precision': 0.6259708737864078,
  'recall': 0.4360838687859317,
  'f1-score': 0.5140522224436914,
  'support': 5914.0},
 '3': {'precision': 0.484207389749702,
  'recall': 0.4860903380197428,
  'f1-score': 0.4851470368711748,
  'support': 3343.0},
 '4': {'precision': 0.13836948391922213,
  'recall': 0.3854166666666667,
  'f1-score': 0.20363236103467253,
  'support': 480.0},
 'accuracy': 0.45548387096774196,
 'macro avg': {'precision': 0.35316367843785373,
  'recall': 0.4106787916762692,
  'f1-score': 0.36263094341553775,
  'support': 11625.0},
 'weighted avg': {'precision': 0.5154922395649099,
  'recall': 0.45548387096774196,
  'f1-score': 0.4718650025938464,
  'support': 11625.0}}

In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_pred = np.zeros(len(y), dtype=int)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = XGBClassifier(**xgb_params)
    sample_weights = compute_sample_weight("balanced", y_train)
    model.fit(X_train, y_train, sample_weight=sample_weights)
    preds = model.predict(X_val)
    oof_pred[val_idx] = preds
    fold_qwk = qwk_score(y_val, preds)
    print(f"Fold {fold} QWK: {fold_qwk:.4f}")

oof_qwk = qwk_score(y, oof_pred)
print(f"OOF QWK: {oof_qwk:.4f}")

Fold 1 QWK: 0.3819
Fold 2 QWK: 0.3486
Fold 3 QWK: 0.3611
Fold 4 QWK: 0.3712
Fold 5 QWK: 0.3784
OOF QWK: 0.3682


In [13]:
# Balanced Random Forest CV (OOF)
rf_oof_pred = np.zeros(len(y), dtype=int)
X_rf = X.copy()
if len(cat_cols) > 0:
    for col in cat_cols:
        X_rf[col] = X_rf[col].cat.codes

for fold, (train_idx, val_idx) in enumerate(skf.split(X_rf, y), start=1):
    X_train = X_rf.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X_rf.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = BalancedRandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rf_oof_pred[val_idx] = preds
    fold_qwk = qwk_score(y_val, preds)
    print(f"RF Fold {fold} QWK: {fold_qwk:.4f}")

rf_oof_qwk = qwk_score(y, rf_oof_pred)
print(f"RF OOF QWK: {rf_oof_qwk:.4f}")

save_classification_report(
    y,
    rf_oof_pred,
    model_name="rf_balanced_oof",
    seed=RANDOM_STATE,
    config="rf_balanced",
    columns=X.columns,
    notes="BalancedRandomForest OOF",
    extra_metrics={"qwk": rf_oof_qwk},
)

# LightGBM CV (OOF)
lgbm_oof_pred = np.zeros(len(y), dtype=int)
cat_feature = list(cat_cols) if len(cat_cols) > 0 else "auto"

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = LGBMClassifier(**lgbm_params)
    model.fit(X_train, y_train, categorical_feature=cat_feature)
    preds = model.predict(X_val)
    lgbm_oof_pred[val_idx] = preds
    fold_qwk = qwk_score(y_val, preds)
    print(f"LGBM Fold {fold} QWK: {fold_qwk:.4f}")

lgbm_oof_qwk = qwk_score(y, lgbm_oof_pred)
print(f"LGBM OOF QWK: {lgbm_oof_qwk:.4f}")

save_classification_report(
    y,
    lgbm_oof_pred,
    model_name="lgbm_balanced_oof",
    seed=RANDOM_STATE,
    config="lgbm_balanced",
    columns=X.columns,
    notes="LightGBM OOF",
    extra_metrics={"qwk": lgbm_oof_qwk},
)

RF Fold 1 QWK: 0.3497
RF Fold 2 QWK: 0.3263
RF Fold 3 QWK: 0.3321
RF Fold 4 QWK: 0.3453
RF Fold 5 QWK: 0.3404
RF OOF QWK: 0.3388
LGBM Fold 1 QWK: 0.3387
LGBM Fold 2 QWK: 0.3369
LGBM Fold 3 QWK: 0.3397
LGBM Fold 4 QWK: 0.3460
LGBM Fold 5 QWK: 0.3443
LGBM OOF QWK: 0.3411


{'0': {'precision': 0.5,
  'recall': 0.04018912529550828,
  'f1-score': 0.07439824945295405,
  'support': 846.0},
 '1': {'precision': 0.40159377316530764,
  'recall': 0.25206467372339186,
  'f1-score': 0.3097262917172872,
  'support': 8597.0},
 '2': {'precision': 0.5895740072202166,
  'recall': 0.6904085497835498,
  'f1-score': 0.6360195036842023,
  'support': 29568.0},
 '3': {'precision': 0.4875658528295474,
  'recall': 0.5149267125336524,
  'f1-score': 0.5008729050279329,
  'support': 16715.0},
 '4': {'precision': 0.5052356020942408,
  'recall': 0.0804837364470392,
  'f1-score': 0.13884892086330936,
  'support': 2398.0},
 'accuracy': 0.5404824169017962,
 'macro avg': {'precision': 0.4967938470618625,
  'recall': 0.3156145595566283,
  'f1-score': 0.33197317414913724,
  'support': 58124.0},
 'weighted avg': {'precision': 0.5276519874278678,
  'recall': 0.5404824169017962,
  'f1-score': 0.5202073644903539,
  'support': 58124.0}}